<a href="https://colab.research.google.com/github/JSVJ/hybrid-rag-eval/blob/main/hybrid_rag_ingestion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Initialization

In [ ]:
def check_credentials(user="JSVJ", repo="hybrid-rag-eval"):
    """Checks GITHUB_TOKEN and GOOGLE_API_KEY. Prints results, never the secrets."""
    import sys, requests
    from google.colab import userdata

    print(f"Python {sys.version.split()[0]}")
    all_ok = True

    def get_secret(name):
        try:
            return userdata.get(name)
        except Exception as e:
            print(f"[FAIL] {name}: {type(e).__name__} (add it in Secrets and turn on Notebook access)")
            return None

    # --- GitHub token ---
    gh = get_secret("GITHUB_TOKEN")
    if gh:
        r = requests.get(
            f"https://api.github.com/repos/{user}/{repo}",
            headers={"Authorization": f"Bearer {gh}", "Accept": "application/vnd.github+json"},
            timeout=20,
        )
        if r.status_code == 200:
            perms = r.json().get("permissions")
            push = perms.get("push") if isinstance(perms, dict) else None
            expiry = r.headers.get("github-authentication-token-expiration", "not reported")
            print(f"[ OK ] GitHub: repo {user}/{repo} reachable | push permission: {push} | token expires: {expiry}")
            if push is False:
                print("       Token is read-only: set Contents to 'Read and write' on the token.")
                all_ok = False
        else:
            print(f"[FAIL] GitHub: HTTP {r.status_code} (token invalid, expired, or not scoped to this repo)")
            all_ok = False
    else:
        all_ok = False

    # --- Gemini API key (lists models: no generation, no token cost) ---
    gk = get_secret("GOOGLE_API_KEY")
    if gk:
        r = requests.get(
            "https://generativelanguage.googleapis.com/v1beta/models",
            headers={"x-goog-api-key": gk},
            params={"pageSize": 5},
            timeout=20,
        )
        if r.status_code == 200:
            names = [m["name"] for m in r.json().get("models", [])]
            print(f"[ OK ] Gemini: key valid | sample models: {names}")
        else:
            msg = r.json().get("error", {}).get("message", "") if r.headers.get("content-type", "").startswith("application/json") else ""
            print(f"[FAIL] Gemini: HTTP {r.status_code} {msg}")
            all_ok = False
    else:
        all_ok = False

    print("\nAll checks passed." if all_ok else "\nSome checks failed. See above.")
    return all_ok

check_credentials()

## Ingestion

In [ ]:
# folder where we need to keep the files (can update, delete, upload new)
# folder = /content/drive/MyDrive/Colab Notebooks/RAG Project/downloaded_files
from pathlib import Path
DATA_DIR = Path(r"/content/drive/MyDrive/Colab Notebooks/RAG Project/downloaded_files")
if DATA_DIR.exists(): print('Folder exists')
files = sorted(DATA_DIR.glob('*.pdf'))

for p in files:
    size_kb = p.stat().st_size / 1024
    print(f"{p.name:<30} {size_kb:>8.0f} KB")

In [ ]:
# finger printing each file using SHA-256
# right now, the pipeline doesn't use hash. This is for verification purposes only.
import hashlib

for file in files:
  digest = hashlib.sha256(file.read_bytes()).hexdigest()
  print(f"{digest[:12]} {file.name}") # just for comparision

In [ ]:
# Creating a Stable ID
# In production this must not be hard coded, this has to be automatically generated using code.
FILE_TO_DOC_ID = {
    "Attention Is All You Need.pdf": "attention",
    "Retrieval-Augmented Generation for Knowledge Intensive NLP Tasks.pdf": "rag",
    "BERT.pdf": "bert",
    "Chain-of-Thought Prompting.pdf": "cot",
    "Lost in the Middle - How Language Models Use Long Context.pdf": "lost_in_middle",
    "Artificial Intelligence Risk Management.pdf": "nist_ai_rmf",
}

found = {p.name for p in files}
mapped = set(FILE_TO_DOC_ID)

print('found but not in mapped', found - mapped)
print('mapped but not in found', mapped - found)

In [ ]:
# installing the PDF libraries
!pip install pymupdf

In [ ]:
# Convert Hetrogeneous data formats (word, excel, csv, image, logs etc) into PDF

In [ ]:
# parsing all pdfs into texts
import pymupdf

pages = [] # one dictionary per page

for file in files:
  # get the stable ID
  document_id = FILE_TO_DOC_ID[file.name]
  document = pymupdf.open(file)
  for page_number, page in enumerate(document, start = 1):
    pages.append({
        "document_id": document_id,
        "page_number": page_number,
        "text": page.get_text()
    })
  document.close()
print(f"Total pages: {len(pages)}")


## Chunking

In [ ]:
# Chunking - Start with Fixed size but go to Structure aware chunking
CHUNK_SIZE = 1000 # characters per chunk
OVERLAP = 200 # characters shared with previous chunk so that we don't cut anything half way
STEP = CHUNK_SIZE - OVERLAP

chunks = []

for page in pages:
  text = page['text']
  chunk_number = 0
  for start in range(0, len(text), STEP):
    piece = text[start:start + CHUNK_SIZE]
    if len(piece) >= 100: # skip tiny left overs
      chunks.append({
                "chunk_id": f"{page['document_id']}:p{page['page_number']}:c{chunk_number}",
                "document_id": page["document_id"],
                "page": page["page_number"],
                "text": piece,
            })
      chunk_number += 1
    if start + CHUNK_SIZE >= len(text): # end of the page window
      break

In [ ]:
# TODO: Structure aware chunking will come here

## Embeddings

In [ ]:
# install sentence transformers from hugging face
!pip install -q sentence-transformers
!pip freeze | grep -i -E "^sentence-transformers=="

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer("BAAI/bge-small-en-v1.5")

In [ ]:
print("Max tokens per input:", model.max_seq_length)
print("Embedding size:", model.get_embedding_dimension())

texts = [chunk['text'] for chunk in chunks]

token_counts = [len(model.tokenizer.encode(text)) for text in texts]
print(f'Token per chunk (avg): {sum(token_counts)/len(token_counts)}, Max Token per chunk: {max(token_counts)}')

embeddings = model.encode(texts, batch_size = 32, normalize_embeddings = True, show_progress_bar = True)
print(f"Embedding shape: {embeddings.shape}")